# CNN - Mamografias CBIS-DDSM

Treino da CNN (MobileNetV2) + predição e explicabilidade Grad-CAM com o
modelo salvo em `models/cnn/mobilenet_mammo.pth`. Este notebook segue o
padrão dos demais: usa os módulos `src/cnn/` da API.

In [1]:
# Celula 1 - Imports
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import torch
from src.cnn.dataset import create_dataloaders
from src.cnn.predict import MammoPredictor
from src.cnn.train import train_cnn

print('PyTorch:', torch.__version__)
print('CUDA disponivel:', torch.cuda.is_available())

C:\Users\ricoi\POSTECH\tech-challenge-fase1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0+cpu
CUDA disponivel: False


In [2]:
# Celula 2 - Visualizar amostras do dataset
train_loader, val_loader, test_loader = create_dataloaders(
    base_dir='data/images/cbis-ddsm',
    img_size=(224, 224), batch_size=9
)
print('Treino:', len(train_loader.dataset),
      '| Validacao:', len(val_loader.dataset),
      '| Teste:', len(test_loader.dataset))
print('Contagem por classe (treino):', train_loader.dataset.class_counts)

batch_imgs, batch_labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, ax in enumerate(axes.flatten()):
    img = batch_imgs[i].permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # denormalize
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    label = 'Maligno' if batch_labels[i] == 1 else 'Benigno'
    ax.set_title(label, color='red' if label == 'Maligno' else 'green')
    ax.axis('off')
plt.tight_layout()
plt.show()

Treino: 375 | Validacao: 94 | Teste: 90
Contagem por classe (treino): {'benign': 207, 'malignant': 168}


C:\Users\ricoi\AppData\Local\Temp\ipykernel_3288\3092319415.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Celula 3 - Modelo treinado (checkpoint consumido pela API)
model_path = 'models/cnn/mobilenet_mammo.pth'
print('Modelo salvo em:', model_path)
print('Existe:', os.path.exists(model_path))

if os.path.exists(model_path):
    predictor = MammoPredictor(model_path=model_path)
    print('\nMetadados do checkpoint:')
    for k, v in predictor.get_checkpoint_meta().items():
        print(f'  {k}: {v}')

Modelo salvo em: models/cnn/mobilenet_mammo.pth
Existe: True



Metadados do checkpoint:
  architecture: MobileNetV2
  img_size: [224, 224]
  test_metrics: {'test_accuracy': 0.45555555555555555, 'test_auc': 0.5158441558441559, 'test_recall': 0.8, 'test_precision': 0.4, 'test_f1': 0.5333333333333333}


In [4]:
# Celula 4 - Predicao de exemplo com o modelo salvo
sample_path = None
if os.path.exists(model_path):
    candidates = []
    for label in ('malignant', 'benign'):
        d = Path('data/images/cbis-ddsm/test') / label
        candidates += sorted(d.glob('*.jpg')) + sorted(d.glob('*.png'))
    if candidates:
        sample_path = candidates[0]
        print('Amostra:', sample_path.name)
        result = predictor.predict_from_bytes(sample_path.read_bytes())
        for k, v in result.items():
            print(f'  {k}: {v}')

Amostra: 015a45163af9a37e_1-009.jpg


  prediction: malignant
  probability_malignant: 0.6272263526916504
  probability_benign: 0.3727736473083496
  confidence: 0.6272263526916504
  model_used: mobilenet_mammo_pytorch


In [5]:
# Celula 5 - Grad-CAM para explicabilidade visual
if os.path.exists(model_path) and sample_path is not None:
    from PIL import Image
    img = Image.open(sample_path).convert('RGB')
    img_array = np.array(img)
    heatmap = predictor.compute_gradcam(img_array)
    prediction = predictor.predict(img_array)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(img_array)
    axes[0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    axes[2].imshow(img_array)
    axes[2].imshow(heatmap, cmap='jet', alpha=0.5)
    axes[2].set_title(f"Overlay ({prediction['prediction']})", fontsize=12, fontweight='bold')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

C:\Users\ricoi\AppData\Local\Temp\ipykernel_3288\2520898132.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Celula 6 - (OPCIONAL) Retreinar a CNN
# O treino completo (feature extraction + fine-tuning) e pesado e o
# checkpoint ja existe. Para treinar do zero, mude TREINAR para True.
# Use pretrained=True para transfer learning com pesos ImageNet (download).
TREINAR = False

if TREINAR:
    model, metrics = train_cnn(
        epochs=10,
        finetune_epochs=3,
        batch_size=16,
        img_size=(224, 224),
        data_dir='data/images/cbis-ddsm',
        experiment_name='mammo_cnn',
        pretrained=False,
        patience=5,
    )
    print('Metricas finais:', metrics)
else:
    print('Treino pulado (TREINAR = False). O checkpoint existente sera usado pela API.')

Treino pulado (TREINAR = False). O checkpoint existente sera usado pela API.
